In [1]:
import nltk
import pandas as pd
import numpy as np
import string
import sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity

# Preprocessing du dataframe

In [ ]:
df = pd.read_csv("train_submission.csv")

X = df['Text'].to_list()
y = df['Label'].to_list()
labels = df.groupby('Label').first().reset_index()['Label'].to_list()

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.1)

In [3]:
def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

def preprocess_corpus(corpus):
    words = []
    
    for sentence in corpus:
        # Split sentence into words
        sentence = remove_punctuation(sentence)
        tokens = sentence.split()
        
        # Format each word with a leading space and 4 trailing spaces
        formatted_tokens = [f" {word} " for word in tokens]
        
        # Add to final list
        words.extend(formatted_tokens)
    
    return words

Separating texts by language

In [4]:
texts_dict = {}

for label in tqdm(labels):
    X_lang = [X_train[i] for i,x in enumerate(y_train) if x == label]
    texts_dict[label] = X_lang

100%|██████████| 389/389 [00:01<00:00, 328.56it/s]


Ranking the K most frequent n-grams in every language

In [5]:
def get_K_ngrams(corpus,K=200):
    X_preprocessed = preprocess_corpus(corpus)
    vectorizer = CountVectorizer(analyzer="char",ngram_range=(1,3),lowercase=True,max_features= K)
    ng_count = np.asarray(vectorizer.fit_transform(X_preprocessed).mean(axis=0)).flatten()
    ng = vectorizer.get_feature_names_out()

    #sort the N-grams by frequency in deescending order
    sorted_indices = np.argsort(-ng_count)
    ngram_df = pd.DataFrame({"ngram":ng[sorted_indices],"rank":list(range(len(ng)))})
    
    return ngram_df

In [6]:
n_grams = {}
vectorizer_dict = {}
for label in tqdm(labels):
    n_grams[label] = get_K_ngrams(texts_dict[label])

100%|██████████| 389/389 [00:18<00:00, 20.61it/s]


In [7]:
def compute_rank_differences(list_one, list_two):
    # Create DataFrames for rankings
    df_one = pd.DataFrame({'word': list_one, 'rank_one': range(len(list_one))})
    df_two = pd.DataFrame({'word': list_two, 'rank_two': range(len(list_two))})

    # Merge on 'word' to align rankings
    merged = pd.merge(df_two, df_one, on='word', how='left')

    # Compute rank differences (NaN if word is missing from list_one)
    merged['rank_difference'] = merged['rank_two'] - merged['rank_one']

    return merged[['word', 'rank_difference']].fillna(len(merged))['rank_difference'].abs().sum()

# Example ranked lists
list_one = ["apple", "banana", "cherry", "date", "elderberry"]
list_two = ["banana", "apple", "elderberry", "fig", "cherry"]

# Compute rank differences without loops
result = compute_rank_differences(list_one, list_two)

# Display results
print(result)

11.0


In [8]:
y_pred = []
k = 1
for t in tqdm(X_test):
    # Computing the n_grams of the text
    guess = "ZZZ"
    n_grams_df = get_K_ngrams([t])
    best_sim = 1e10
    for label in labels:
        sim = compute_rank_differences(list(n_grams[label]["ngram"]),list(n_grams_df["ngram"]))
        if sim < best_sim:
            best_sim = sim
            guess = label

    y_pred.append(guess)

    if k % 100 == 0:
        print("Accuracy: ", accuracy_score(y_test[:k],y_pred))

    k += 1

  5%|▌         | 100/1943 [02:17<42:43,  1.39s/it]

Accuracy:  0.73


  5%|▌         | 106/1943 [02:27<42:35,  1.39s/it]


KeyboardInterrupt: 

In [73]:
print("Accuracy: ", accuracy_score(y_test,y_pred))

Accuracy:  0.46937725167267114


In [107]:
texts_dict['msa']

['Sesiapa yang berusaha pasti akan beroleh ganjaranya.',
 'Casas de Reina merupakan sebuah kawasan perbandaran yang terletak di Sepanyol dalam wilayah Badajoz  Extremadura.',
 'Penyembur Cat tanpa Penyaman Penyembur tdk berhati-hati Semburan cat Penyembur cat Peralatan tidak bernafas Senapang semburan Penyembur Aaa tanpa Penyaman Petua Penyembur Cat tanpa Penyaman',
 'Set bagi set aksara piawai untuk artikel yang dipos dengan profil ini',
 "Li'l Dice  apa khabar? Selamat hari jadi.",
 'Bagaimana Latihan Meningkatkan Bahagian Spiritual Penjagaan Elder',
 'Betul kata awak. Bukan gaya saya.',
 'Di sini anda boleh mencari produk berkaitan dalam Tangan Logam Pengesan Logam  kami adalah pengeluar profesional Tangan Logam Pengesan Logam Pengimbas Badan Pegang Tangan Pengesan Logam Pegang Tangan Universal Pengesan Logam Aluminium  . Kami memberi tumpuan kepada pembangunan produk eksport antarabangsa  pengeluaran dan jualan. Kami telah meningkatkan proses kawalan kualiti Tangan Logam Pengesan L

In [101]:
list(zip(y_pred[:400],y_test[:400]))

[('guc', 'guc'),
 ('chk', 'chk'),
 ('arb', 'arb'),
 ('crh', 'crh'),
 ('mad', 'msa'),
 ('kea', 'kea'),
 ('bih', 'bih'),
 ('sqi', 'sqi'),
 ('scn', 'scn'),
 ('que', 'scn'),
 ('ctu', 'ctu'),
 ('xav', 'xav'),
 ('sna', 'ssw'),
 ('kos', 'kos'),
 ('nch', 'nch'),
 ('mal', 'mal'),
 ('twi', 'aka'),
 ('urd', 'tgk'),
 ('ory', 'ori'),
 ('umb', 'umb'),
 ('dtp', 'dtp'),
 ('kac', 'kac'),
 ('ast', 'ast'),
 ('cbk', 'ile'),
 ('nds', 'cbk'),
 ('kos', 'kos'),
 ('orm', 'orm'),
 ('ace', 'ace'),
 ('swh', 'swa'),
 ('oss', 'oss'),
 ('mco', 'mco'),
 ('pam', 'pam'),
 ('ksd', 'ksd'),
 ('twi', 'aka'),
 ('lua', 'lua'),
 ('kek', 'kek'),
 ('bel', 'bel'),
 ('gsw', 'gsw'),
 ('mzn', 'mzn'),
 ('bcl', 'bcl'),
 ('ast', 'ast'),
 ('fin', 'est'),
 ('srn', 'srn'),
 ('cmn', 'lzh'),
 ('urd', 'tgk'),
 ('rmy', 'wol'),
 ('ara', 'hau'),
 ('lua', 'lua'),
 ('lus', 'lus'),
 ('nya', 'nbl'),
 ('bzj', 'bzj'),
 ('tah', 'dtp'),
 ('ven', 'ven'),
 ('tah', 'tah'),
 ('oci', 'oci'),
 ('fra', 'fra'),
 ('mau', 'mau'),
 ('bos', 'hbs'),
 ('alt', 'alt'